# Lab 9: Genetički Algoritmi


## Setup


In [ ]:
import sys, subprocess, importlib

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyeasyga"],
               capture_output=True)
try:
    from pyeasyga.pyeasyga import GeneticAlgorithm
    print("pyeasyga ucitan.")
except ImportError:
    print("Koristim drop-in GeneticAlgorithm (kompatibilan interfejs).")
    import random
    from operator import attrgetter

    class GeneticAlgorithm:
        def __init__(self, seed_data, population_size=50, generations=100,
                     crossover_probability=0.8, mutation_probability=0.2,
                     elitism=True, maximise_fitness=True):
            self.seed_data         = seed_data
            self.population_size   = population_size
            self.generations       = generations
            self.crossover_probability = crossover_probability
            self.mutation_probability  = mutation_probability
            self.elitism           = elitism
            self.maximise_fitness  = maximise_fitness
            self.tournament_size   = max(2, population_size // 10)
            self.create_individual = lambda d: [random.randint(0,1) for _ in range(10)]
            self.fitness_function  = None
            self.crossover_function = self._spx
            self.mutate_function   = self._flip
            self.selection_function = self._tournament
            self.current_generation = []

        class _C:
            __slots__ = ('genes','fitness')
            def __init__(self, g): self.genes=list(g); self.fitness=0

        def _spx(self,a,b):
            p=random.randrange(1,len(a)); return a[:p]+b[p:], b[:p]+a[p:]
        def _flip(self,ind):
            i=random.randrange(len(ind)); ind[i]=1-ind[i]
        def _best(self,pop):
            return (max if self.maximise_fitness else min)(pop, key=attrgetter('fitness'))
        def _tournament(self, pop):
            t = random.sample(pop, min(self.tournament_size, len(pop)))
            return (max if self.maximise_fitness else min)(t, key=attrgetter('fitness'))
        def _eval(self, pop):
            for c in pop: c.fitness = self.fitness_function(c.genes, self.seed_data)
        def _breed(self, pop):
            nxt = []
            if self.elitism:
                e = self._C(self._best(pop).genes)
                e.fitness = self._best(pop).fitness
                nxt.append(e)
            while len(nxt) < self.population_size:
                p1 = self._C(self.selection_function(pop).genes)
                p2 = self._C(self.selection_function(pop).genes)
                if random.random() < self.crossover_probability:
                    g1,g2 = self.crossover_function(list(p1.genes), list(p2.genes))
                else:
                    g1,g2 = list(p1.genes), list(p2.genes)
                c1,c2 = self._C(g1), self._C(g2)
                if random.random() < self.mutation_probability: self.mutate_function(c1.genes)
                if random.random() < self.mutation_probability: self.mutate_function(c2.genes)
                nxt.append(c1)
                if len(nxt) < self.population_size: nxt.append(c2)
            return nxt
        def run(self):
            self.current_generation = [self._C(self.create_individual(self.seed_data))
                                       for _ in range(self.population_size)]
            self._eval(self.current_generation)
            for _ in range(self.generations):
                self.current_generation = self._breed(self.current_generation)
                self._eval(self.current_generation)
        def best_individual(self):
            b = self._best(self.current_generation)
            return (b.fitness, b.genes)
        def last_generation(self):
            return [(c.fitness, c.genes) for c in self.current_generation]


## Primjer korištenja: Drop-Wave funkcija

Minimizacija Drop-Wave funkcije:
$$f(x_1, x_2) = \frac{-1 - \cos\left(12\sqrt{x_1^2 + x_2^2}\right)}{0.5(x_1^2 + x_2^2) + 2}$$
Globalni minimum: $f(0,0) = -1$


In [ ]:
import random
import numpy as np
import math

def criteria_function(x):
    x1 = x[0]; x2 = x[1]
    return (-1 - np.cos(12*np.sqrt(x1**2 + x2**2))) / (0.5*(x1**2 + x2**2) + 2)

# x is in range [-5, 5], y is in range [-5, 5]
RANGE = np.array([[-5, 5], [-5, 5]])

ga = GeneticAlgorithm(seed_data=criteria_function,
                      population_size=30,
                      generations=200,
                      crossover_probability=0.8,
                      mutation_probability=0.02,
                      elitism=True,
                      maximise_fitness=False)

# first half => x, second half => y
BINARY_CHROMOSOME_LENGTH = 26

def create_individual(data):
    return [random.randint(0, 1) for _ in range(BINARY_CHROMOSOME_LENGTH)]

ga.create_individual = create_individual

# two-point crossover
def crossover(parent_1, parent_2):
    crossover_index1 = random.randrange(1, len(parent_1)-1)
    crossover_index2 = random.randrange(crossover_index1+1, len(parent_1))
    child_1 = parent_1[:crossover_index1] + parent_2[crossover_index1:crossover_index2] + parent_1[crossover_index2:]
    child_2 = parent_2[:crossover_index1] + parent_1[crossover_index1:crossover_index2] + parent_2[crossover_index2:]
    return child_1, child_2

ga.crossover_function = crossover

# randomly select a bit and invert it
def mutate(individual):
    mutate_index = random.randrange(len(individual))
    if individual[mutate_index] == 0:
        individual[mutate_index] = 1
    else:
        individual[mutate_index] = 0

ga.mutate_function = mutate
ga.tournament_size = 2

def decimal(binary):
    sum = 0
    for i in range(0, len(binary)):
        sum += binary[len(binary)-1-i] * 2**(i)
    return sum

def decode(individual):
    x_binary = individual[:BINARY_CHROMOSOME_LENGTH // 2]
    y_binary = individual[BINARY_CHROMOSOME_LENGTH // 2:]
    x_range = RANGE[0]; y_range = RANGE[1]
    x_min = x_range[0]; x_max = x_range[1]
    y_min = y_range[0]; y_max = y_range[1]
    x_decode = x_min + ((x_max - x_min)*decimal(x_binary))/(2**(len(x_binary)) - 1)
    y_decode = y_min + ((y_max - y_min)*decimal(y_binary))/(2**(len(y_binary)) - 1)
    return x_decode, y_decode

def is_in_range(x, range_x):
    return x >= range_x[0] and x <= range_x[1]

def fitness(individual, data):
    x_decode, y_decode = decode(individual)
    while not (is_in_range(x_decode, RANGE[0]) and is_in_range(y_decode, RANGE[1])):
        individual = create_individual(data)
        x_decode, y_decode = decode(individual)
    fitness = data([x_decode, y_decode])
    return fitness

ga.fitness_function = fitness

random.seed(42)
ga.run()

print("Best individual fitness and best individual binary chromosome:")
print(ga.best_individual())

print("Best individual decoded:")
print(decode(ga.best_individual()[1]))

print("Best individual function value:")
print(criteria_function(decode(ga.best_individual()[1])))

print("\n---------------------------\n")

print("Last generation:")
for individual in ga.last_generation():
    print(individual)


## Zadatak 1: Minimizacija Rastriginove funkcije

Modifikacija prethodnog primjera za minimizaciju Rastriginove funkcije:
$$f(x_1, x_2) = 20 + x_1^2 + x_2^2 - 10(\cos(2\pi x_1) + \cos(2\pi x_2))$$
Domen: $x_1, x_2 \in [-5.12,\, 5.12]$,  globalni minimum: $f(0,0)=0$


In [ ]:
import matplotlib.pyplot as plt

# --- Kriterijalna funkcija - Rastrigin ---
def criteria_function_r(x):
    x1 = x[0]; x2 = x[1]
    return 20 + x1**2 + x2**2 - 10*(math.cos(2*math.pi*x1) + math.cos(2*math.pi*x2))

RANGE_R = np.array([[-5.12, 5.12], [-5.12, 5.12]])

# Dekodiranje binarnog hromosoma u realne koordinate
def decimal(binary):
    s = 0
    for i in range(len(binary)):
        s += binary[len(binary)-1-i] * 2**i
    return s

def decode_r(individual, clen):
    x_bin = individual[:clen // 2]
    y_bin = individual[clen // 2:]
    x_min, x_max = RANGE_R[0]
    y_min, y_max = RANGE_R[1]
    x = x_min + (x_max - x_min) * decimal(x_bin) / (2**len(x_bin) - 1)
    y = y_min + (y_max - y_min) * decimal(y_bin) / (2**len(y_bin) - 1)
    return x, y

def fitness_r(individual, data):
    clen = len(individual)
    x, y = decode_r(individual, clen)
    return data([x, y])

def crossover_r(parent_1, parent_2):
    i1 = random.randrange(1, len(parent_1)-1)
    i2 = random.randrange(i1+1, len(parent_1))
    c1 = parent_1[:i1] + parent_2[i1:i2] + parent_1[i2:]
    c2 = parent_2[:i1] + parent_1[i1:i2] + parent_2[i2:]
    return c1, c2

def mutate_r(individual):
    idx = random.randrange(len(individual))
    individual[idx] = 1 - individual[idx]

print("Rastrigin funkcija i pomocne funkcije ucitane.")


### a) Vizualizacija Rastriginove funkcije


In [ ]:
x_vals = np.linspace(-5.12, 5.12, 300)
y_vals = np.linspace(-5.12, 5.12, 300)
X, Y = np.meshgrid(x_vals, y_vals)
Z = 20 + X**2 + Y**2 - 10*(np.cos(2*np.pi*X) + np.cos(2*np.pi*Y))

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.85)
ax1.set_title('Rastrigin - 3D')
ax1.set_xlabel('x1'); ax1.set_ylabel('x2'); ax1.set_zlabel('f')

ax2 = fig.add_subplot(122)
cp = ax2.contourf(X, Y, Z, 50, cmap='viridis')
plt.colorbar(cp, ax=ax2)
ax2.set_title('Rastrigin - konturni dijagram')
ax2.set_xlabel('x1'); ax2.set_ylabel('x2')
plt.tight_layout(); plt.show()

print("Oblik funkcije: Da, funkcija je MULTIMODALNA.")
print("Ima veliki broj lokalnih minimuma (otprilike 100+ u datom domenu).")
print("Globalni minimum je u tacki (0, 0) sa vrijednoscu f = 0.")


### b) Promjena dužine hromosoma na 6


In [ ]:
print(f"{'clen':>6} {'bits/dim':>9} {'tacaka/dim':>11} {'x1':>9} {'x2':>9} {'f':>10}")
print("-" * 60)

for clen in [6, 10, 20, 26]:
    bits_per_dim = clen // 2
    n_points = 2**bits_per_dim

    ga = GeneticAlgorithm(seed_data=criteria_function_r,
                          population_size=30,
                          generations=200,
                          crossover_probability=0.8,
                          mutation_probability=0.02,
                          elitism=True,
                          maximise_fitness=False)

    ga.create_individual = lambda d, c=clen: [random.randint(0,1) for _ in range(c)]
    ga.crossover_function = crossover_r
    ga.mutate_function    = mutate_r
    ga.fitness_function   = fitness_r
    ga.tournament_size    = 2

    random.seed(42)
    ga.run()
    x, y = decode_r(ga.best_individual()[1], clen)
    f = criteria_function_r([x, y])
    print(f"{clen:>6} {bits_per_dim:>9} {n_points:>11} {x:>9.4f} {y:>9.4f} {f:>10.6f}")

print()
print("Zakljucak: Sa clen=6 svaka dimenzija ima samo 2^3=8 mogucih vrijednosti.")
print("Diskretizacija je gruba (~1.46 po koraku) -> manja preciznost rjesenja.")
print("Sa clen=26: 2^13=8192 vrijednosti -> mnogo finija diskretizacija (~0.00125).")


### c) Utjecaj parametra `mutation_probability`


In [ ]:
CLEN = 26
print(f"{'mut_prob':>10} {'prosj. f (5 pokretanja)':>25}")
print("-" * 38)
results_mut = {}
for mp in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.3, 0.7, 0.99]:
    vals = []
    for seed in range(5):
        random.seed(seed)
        ga = GeneticAlgorithm(seed_data=criteria_function_r,
                              population_size=30, generations=200,
                              crossover_probability=0.8,
                              mutation_probability=mp,
                              elitism=True, maximise_fitness=False)
        ga.create_individual = lambda d: [random.randint(0,1) for _ in range(CLEN)]
        ga.crossover_function = crossover_r
        ga.mutate_function    = mutate_r
        ga.fitness_function   = fitness_r
        ga.tournament_size    = 2
        ga.run()
        x, y = decode_r(ga.best_individual()[1], CLEN)
        vals.append(criteria_function_r([x, y]))
    avg = sum(vals)/len(vals)
    results_mut[mp] = avg
    print(f"{mp:>10.3f} {avg:>25.6f}")

plt.figure(figsize=(8,4))
plt.semilogx(list(results_mut.keys()), list(results_mut.values()), 'o-', color='tomato')
plt.xlabel('mutation_probability (log skala)'); plt.ylabel('Prosjecna f vrijednost')
plt.title('Utjecaj mutation_probability na Rastrigin')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print()
print("Zakljucak:")
print("  ~0:    premalo istrazivanja, GA zapada u lokalni optimum")
print("  0.02:  optimalna vrijednost za ovaj problem (balans eksploatacija/eksploracija)")
print("  >0.3:  previse mutacije -> pretvorba u slucajnu pretragu, rezultati se pogorsavaju")


### d) Utjecaj parametara `generations` i `population_size`


In [ ]:
CLEN = 26
configs = [
    ( 5,  50,  "mala pop,    malo gen"),
    ( 5,  500, "mala pop,    puno gen"),
    (15,  100, "srednja pop, umjereno gen"),
    (30,  200, "default (pop=30, gen=200)"),
    (50,   50, "veca pop,    malo gen"),
    (50,  200, "veca pop,    normalno gen"),
    (100,  50, "velika pop,  malo gen"),
    (100, 200, "velika pop,  normalno gen"),
]
print(f"{'pop':>5} {'gen':>5} {'prosj. f':>12}  Opis")
print("-" * 55)
for pop, gen, opis in configs:
    vals = []
    for seed in range(5):
        random.seed(seed)
        ga = GeneticAlgorithm(seed_data=criteria_function_r,
                              population_size=pop, generations=gen,
                              crossover_probability=0.8, mutation_probability=0.02,
                              elitism=True, maximise_fitness=False)
        ga.create_individual = lambda d: [random.randint(0,1) for _ in range(CLEN)]
        ga.crossover_function = crossover_r
        ga.mutate_function    = mutate_r
        ga.fitness_function   = fitness_r
        ga.tournament_size    = 2
        ga.run()
        x, y = decode_r(ga.best_individual()[1], CLEN)
        vals.append(criteria_function_r([x, y]))
    avg = sum(vals)/len(vals)
    print(f"{pop:>5} {gen:>5} {avg:>12.6f}  {opis}")

print()
print("Zakljucak: Za manje populacije je potrebno vise generacija da bi se")
print("dostiglo isto rjesenje. Veca populacija brze konvergira, ali svaka")
print("generacija traje duze. Optimum ovisi o racunskim resursima.")


### e) `population_size=5`, bez elitizma


In [ ]:
CLEN = 26
print(f"{'pop':>5} {'elitizam':>10} {'prosj. f (10 run)':>18}")
print("-" * 38)
for pop, elitism in [(5, True), (5, False), (30, True), (30, False)]:
    vals = []
    for seed in range(10):
        random.seed(seed)
        ga = GeneticAlgorithm(seed_data=criteria_function_r,
                              population_size=pop, generations=200,
                              crossover_probability=0.8, mutation_probability=0.02,
                              elitism=elitism, maximise_fitness=False)
        ga.create_individual = lambda d: [random.randint(0,1) for _ in range(CLEN)]
        ga.crossover_function = crossover_r
        ga.mutate_function    = mutate_r
        ga.fitness_function   = fitness_r
        ga.tournament_size    = 2
        ga.run()
        x, y = decode_r(ga.best_individual()[1], CLEN)
        vals.append(criteria_function_r([x, y]))
    avg = sum(vals)/len(vals)
    label = "Da" if elitism else "Ne"
    print(f"{pop:>5} {label:>10} {avg:>18.6f}")

print()
print("Zakljucak: population_size=5 bez elitizma daje znacajno losije rezultate.")
print("Mala populacija bez elitizma = geneticki drift + gubici dobrih rjesenja.")
print("Dobro rjesenje se moze izgubiti ukrstanjem/mutacijom i nema garancije da")
print("ce prezivjeti u sljedecoj generaciji.")


### f) Rulet-točak selekcija vs selekcija na bazi ranga


In [ ]:
from operator import attrgetter

CLEN = 26

# Rulet-tocak selekcija za minimizaciju (invertujemo fitness)
def roulette_selection(population):
    fits = [ind.fitness for ind in population]
    f_min = min(fits)
    inv = [1.0 / (f - f_min + 1e-10) for f in fits]
    total = sum(inv)
    pick  = random.uniform(0, total)
    cum   = 0.0
    for ind, w in zip(population, inv):
        cum += w
        if cum >= pick:
            return ind
    return population[-1]

# Selekcija na bazi ranga (formula iz PDF, prilagodjena za minimizaciju)
def rank_selection(population, SP=1.5):
    m = len(population)
    # Sortiraj uzlazno (manji fitness = bolji za minimizaciju)
    # -> dodjeli mu veci rank (rank m = najmanji fitness = najbolji)
    sorted_pop = sorted(population, key=attrgetter('fitness'))
    weights = [(2 - SP) + 2*(SP - 1)*(m - 1 - i)/(m - 1) for i in range(m)]
    total = sum(weights)
    pick  = random.uniform(0, total)
    cum   = 0.0
    for ind, w in zip(sorted_pop, weights):
        cum += w
        if cum >= pick:
            return ind
    return sorted_pop[0]

methods = {
    'Turnirska (default, k=2)' : None,
    'Rulet-tocak'               : roulette_selection,
    'Na bazi ranga (SP=1.5)'    : lambda pop: rank_selection(pop, 1.5),
    'Na bazi ranga (SP=2.0)'    : lambda pop: rank_selection(pop, 2.0),
}

print(f"{'Metoda selekcije':>30} {'prosj. f (10 run)':>18}")
print("-" * 52)
results_sel = {}
for name, sel_fn in methods.items():
    vals = []
    for seed in range(10):
        random.seed(seed)
        ga = GeneticAlgorithm(seed_data=criteria_function_r,
                              population_size=30, generations=200,
                              crossover_probability=0.8, mutation_probability=0.02,
                              elitism=True, maximise_fitness=False)
        ga.create_individual = lambda d: [random.randint(0,1) for _ in range(CLEN)]
        ga.crossover_function = crossover_r
        ga.mutate_function    = mutate_r
        ga.fitness_function   = fitness_r
        ga.tournament_size    = 2
        if sel_fn is not None:
            ga.selection_function = sel_fn
        ga.run()
        x, y = decode_r(ga.best_individual()[1], CLEN)
        vals.append(criteria_function_r([x, y]))
    avg = sum(vals)/len(vals)
    results_sel[name] = avg
    print(f"{name:>30} {avg:>18.6f}")

plt.figure(figsize=(9,4))
plt.bar(list(results_sel.keys()), list(results_sel.values()),
        color=['steelblue','tomato','seagreen','orange'])
plt.ylabel('Prosjecna f vrijednost (manja = bolja)')
plt.title('Usporedba metoda selekcije - Rastrigin')
plt.xticks(rotation=12, ha='right')
plt.tight_layout(); plt.show()


## Zadatak 2: Problem 8 kraljica

Pronaći raspored 8 kraljica na šahovskoj ploči 8×8 tako da nijedna kraljica ne napada preostale.
Primjer je preuzet iz dokumentacije `pyeasyga` biblioteke.


In [ ]:
import random

data = [1, 2, 3, 4, 5, 6, 7, 8]

ga_q = GeneticAlgorithm(data,
                         population_size=100,
                         generations=100,
                         crossover_probability=0.8,
                         mutation_probability=0.2,
                         elitism=True,
                         maximise_fitness=False)

# --- (a) Mehanizam kodiranja ---
# Permutacija [1..8]: individual[i] = red kraljice u koloni i
def create_individual(data):
    individual = list(data)
    random.shuffle(individual)
    return individual

ga_q.create_individual = create_individual

# --- (b) Mehanizam selekcije ---
# Turnirska selekcija (default), tournament_size=2, minimizira fitness
ga_q.tournament_size = 2

# --- (c) Fitness funkcija ---
# Broji dijagonalne napade (0 = savrseno rjesenje, minimiziramo)
def fitness_q(individual, data):
    clashes = 0
    for i in range(len(individual)):
        for j in range(i + 1, len(individual)):
            if abs(individual[i] - individual[j]) == abs(i - j):
                clashes += 1
    return clashes

ga_q.fitness_function = fitness_q

# --- (d) Operator ukrstanja: Order Crossover (OX) ---
def crossover_q(parent_1, parent_2):
    n = len(parent_1)
    child_1 = [None] * n
    child_2 = [None] * n
    start = random.randint(0, n - 1)
    end   = random.randint(start, n - 1)
    child_1[start:end+1] = parent_1[start:end+1]
    child_2[start:end+1] = parent_2[start:end+1]
    remaining_1 = [x for x in parent_2 if x not in child_1]
    remaining_2 = [x for x in parent_1 if x not in child_2]
    ptr = 0
    for i in range(n):
        if child_1[i] is None: child_1[i] = remaining_1[ptr]; ptr += 1
    ptr = 0
    for i in range(n):
        if child_2[i] is None: child_2[i] = remaining_2[ptr]; ptr += 1
    return child_1, child_2

ga_q.crossover_function = crossover_q

# --- (e) Operator mutacije: swap ---
def mutate_q(individual):
    mutate_index_1 = random.randrange(len(individual))
    mutate_index_2 = random.randrange(len(individual))
    individual[mutate_index_1], individual[mutate_index_2] = \
        individual[mutate_index_2], individual[mutate_index_1]

ga_q.mutate_function = mutate_q

# Pokretanje
random.seed(42)
ga_q.run()

print("Best individual fitness and best individual (chromosome):")
print(ga_q.best_individual())
best_fit, best_ind = ga_q.best_individual()
print("\nBest individual fitness (clashes):", best_fit)
print("Solution found!" if best_fit == 0 else "No perfect solution this run.")


In [ ]:
import matplotlib.pyplot as plt

def draw_board(individual):
    n = len(individual)
    clashes = fitness_q(individual, data)
    fig, ax = plt.subplots(figsize=(n, n))
    for row in range(n):
        for col in range(n):
            c = 'white' if (row+col)%2==0 else '#b0b0b0'
            ax.add_patch(plt.Rectangle((col, row), 1, 1, color=c))
    for col, row_val in enumerate(individual):
        color = 'blue' if clashes == 0 else 'red'
        ax.text(col+0.5, row_val-0.5, 'Q', ha='center', va='center',
                fontsize=22, fontweight='bold', color=color)
    ax.set_xlim(0, n); ax.set_ylim(0, n)
    ax.set_xticks(range(n+1)); ax.set_yticks(range(n+1))
    ax.set_xticklabels([''] + list('abcdefgh')[:n])
    ax.set_yticklabels(['']+list(range(1,n+1)))
    ax.grid(True, color='black', lw=0.5)
    ax.set_title(f"8-kraljica  |  dijagonalni napadi: {clashes}")
    plt.tight_layout(); plt.show()

draw_board(best_ind)


### a) Mehanizam kodiranja

Koristi se **permutacijsko kodiranje**: jedinka je lista duljine 8 gdje `individual[i]` predstavlja **red** (1–8) u kojemu se nalazi kraljica u **koloni** `i`. Korištenjem permutacije [1..8] garantujemo da se u svakom redu i svakoj koloni nalazi tačno po jedna kraljica — horizontalni i vertikalni napadi su automatski eliminirani. Jedino što ostaje provjeriti su **dijagonalni napadi**.


### b) Mehanizam selekcije

Koristi se **turnirska selekcija** s `tournament_size=2` (postavljeno eksplicitno, default je `population_size // 10`). U svakoj iteraciji se nasumično biraju 2 jedinke i odabire ona s **manjim** fitnessom (jer `maximise_fitness=False` — minimiziramo broj napada). Turnirska selekcija ne zahtijeva transformaciju kriterijalne u fitness funkciju.


### c) Fitness funkcija

Fitness = broj dijagonalnih napada:

```python
def fitness_q(individual, data):
    clashes = 0
    for i in range(len(individual)):
        for j in range(i + 1, len(individual)):
            if abs(individual[i] - individual[j]) == abs(i - j):
                clashes += 1
    return clashes
```

Par kraljica `(i, j)` je na dijagonali ako je razlika kolona jednaka razlici redova: `|i−j| == |individual[i] − individual[j]|`. **Savršeno rješenje ima fitness = 0.**


### d) Operator ukrštanja

Koristi se **Order Crossover (OX)**:
1. Nasumično se odabira segment `[start, end]`
2. Taj segment se kopira direktno iz roditelja 1 u dijete 1 (i obrnuto za dijete 2)
3. Preostale vrijednosti se popunjavaju redoslijedom pojavljivanja u drugom roditelju, preskačući vrijednosti koje već postoje u djetetu

OX čuva **relativni redoslijed** gena i garantuje da dijete ostaje **validna permutacija** — isti set vrijednosti [1..8].


### e) Operator mutacije

Koristi se **swap mutacija**: nasumično se odabiru dvije pozicije i zamjenjuju im se vrijednosti:

```python
def mutate_q(individual):
    mutate_index_1 = random.randrange(len(individual))
    mutate_index_2 = random.randrange(len(individual))
    individual[mutate_index_1], individual[mutate_index_2] = \
        individual[mutate_index_2], individual[mutate_index_1]
```

Swap mutacija čuva **permutacijsku strukturu** — sve vrijednosti [1..8] ostaju prisutne tačno jednom.


### f) Generalizacija na N-kraljica


In [ ]:
# Za N-kraljica jedina izmjena je:
#   data = list(range(1, N+1))  -- umjesto fiksnog [1..8]
# Fitness, crossover i mutate funkcije su genericke i rade za bilo koji N.

def solve_n_queens(N, pop_size=150, generations=300):
    data_n = list(range(1, N + 1))
    ga_n = GeneticAlgorithm(data_n,
                             population_size=pop_size,
                             generations=generations,
                             crossover_probability=0.8,
                             mutation_probability=0.2,
                             elitism=True,
                             maximise_fitness=False)
    ga_n.create_individual  = lambda d: random.sample(d, len(d))
    ga_n.fitness_function   = fitness_q   # ista funkcija, genericka
    ga_n.crossover_function = crossover_q # isti OX crossover
    ga_n.mutate_function    = mutate_q    # isti swap mutate
    ga_n.tournament_size    = 2
    ga_n.run()
    return ga_n.best_individual()

print(f"{'N':>4} {'fitness':>8}  Status         Rjesenje")
print("-" * 65)
for N in [8, 10, 12]:
    random.seed(42)
    fit, ind = solve_n_queens(N)
    status = "SAVRSENO (0 napada)" if fit == 0 else f"{fit} dijag. napada"
    print(f"{N:>4} {fit:>8}  {status:<20} {ind}")

print()
print("Izmjene potrebne za N-kraljica:")
print("  1. data = list(range(1, N+1))    -- promijeniti velicinu")
print("  2. Fitness, crossover i mutate su vec genericke -- nema izmjena")
print("  3. Za veci N povecati population_size i generations")
